In [1]:
import os
import glob
import polars as pl
import numpy as np
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
DATA_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Input Lists
TRAIN_LIST_PATH = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\ProcessedData\DataSplit_Records\Training\ancient_train_list.txt")
TEST_LIST_PATH = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\ProcessedData\DataSplit_Records\Testing\ancient_test_list.txt")

# Output Directories
TRAIN_OUT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Ancient")
TEST_OUT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Ancient")

# Constants
GRID_SIZE = 10
TICK_RATE = 64
EPOCHS_SEC = [0, 10, 20, 30, 40, 50, 60, None]  # None = Round End

# Standard Ancient Bounds to ensure uniform 10x10 grids across all matches
# Extreme values will be clipped to these edges.
#MAP_BOUNDS = {'x_min': -2087, 'x_max': 2147, 'y_min': -1140, 'y_max': 3870}
MAP_BOUNDS = {'x_min':  -2266.97, 'x_max' :    1395.97, 'y_min' :   -2530.58, 'y_max' :    1771.97}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def load_match_list(filepath):
    """Loads match IDs from the provided text file."""
    if not filepath.exists():
        print(f"Warning: List file not found -> {filepath}")
        return set()
    with open(filepath, 'r', encoding='utf-8') as f:
        return set(line.strip() for line in f if line.strip())

def get_channel_path(match_dir, channel_name):
    """Finds channel files handling missing extensions."""
    pattern = str(match_dir / f"{channel_name}*")
    files = glob.glob(pattern)
    return Path(files[0]) if files else None

# ==========================================
# 3. SPATIO-TEMPORAL ANALYSIS (STOMs)
# ==========================================
def process_match(match_dir, out_filepath):
    wf_path = get_channel_path(match_dir, "weapon_fire")
    h_path = get_channel_path(match_dir, "header")
    
    # 3.1) Skip match if required channels are missing
    if not wf_path or not h_path:
        return False
        
    try:
        df_fire = pl.read_parquet(wf_path)
    except Exception:
        return False

    # Standardize Team Codes
    team_map = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}
    df_fire = df_fire.with_columns(
        pl.col("player_team_code").cast(pl.Utf8).replace(team_map).alias("side")
    ).filter(pl.col("side").is_in(["T", "CT"]))

    if df_fire.is_empty():
        return False

    # 3.2) Convert Ticks to Relative Time (0s = first weapon fire of the round)
    df_fire = df_fire.with_columns(
        (pl.col("tick") - pl.col("tick").min().over("round")).alias("rel_tick")
    )

    # Calculate 10x10 Grid Positions (clipping extreme values)
    df_fire = df_fire.with_columns([
        ((pl.col("player_x_pos").clip(MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max']) - MAP_BOUNDS['x_min']) / 
         (MAP_BOUNDS['x_max'] - MAP_BOUNDS['x_min']) * (GRID_SIZE - 0.01)).floor().cast(pl.Int32).alias("gx"),
        ((pl.col("player_y_pos").clip(MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max']) - MAP_BOUNDS['y_min']) / 
         (MAP_BOUNDS['y_max'] - MAP_BOUNDS['y_min']) * (GRID_SIZE - 0.01)).floor().cast(pl.Int32).alias("gy")
    ])

    rounds = df_fire["round"].unique().sort().to_list()
    match_features = []

    # Process STOMs per Round
    for r_id in rounds:
        r_data = df_fire.filter(pl.col("round") == r_id)
        
        # Initialize Feature Vector starting with the round ID
        round_vector = {"round": r_id}
        
        # Track history to identify "new" activity across epochs
        history = {
            "CT": np.zeros((GRID_SIZE, GRID_SIZE), dtype=bool),
            "T": np.zeros((GRID_SIZE, GRID_SIZE), dtype=bool)
        }

        for epoch_idx, sec in enumerate(EPOCHS_SEC):
            limit = sec * TICK_RATE if sec is not None else float('inf')
            epoch_df = r_data.filter(pl.col("rel_tick") <= limit)
            epoch_label = f"e{sec}" if sec is not None else "eEnd"

            for side in ["CT", "T"]:
                side_events = epoch_df.filter(pl.col("side") == side)
                
                # Create boolean mask for cumulative activity up to this epoch
                current_mask = np.zeros((GRID_SIZE, GRID_SIZE), dtype=bool)
                if not side_events.is_empty():
                    for row in side_events.select(["gx", "gy"]).unique().iter_rows():
                        current_mask[row[1], row[0]] = True
                
                # Determine new vs historical activity
                new_activity = current_mask & ~history[side]
                
                # Build STOM State-Grid: 0=None, 1=History, 2=New
                stom_grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
                stom_grid[history[side]] = 1
                stom_grid[new_activity] = 2
                
                # Flatten 10x10 grid to 100 features and assign to vector
                flat_grid = stom_grid.flatten()
                for i, val in enumerate(flat_grid):
                    col_name = f"{side}_{epoch_label}_cell_{i:02d}"
                    round_vector[col_name] = val
                    
                # Update history for the next 10s increment
                history[side] = current_mask.copy()

        match_features.append(round_vector)

    # 4) Save STOM feature vectors as CSV
    if match_features:
        out_df = pl.DataFrame(match_features)
        out_df.write_csv(out_filepath)
        return True
    return False

# ==========================================
# 4. MAIN EXECUTION ROUTINE
# ==========================================
def main():
    # Setup Output Directories
    TRAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)
    TEST_OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Load Train/Test Lists
    train_set = load_match_list(TRAIN_LIST_PATH)
    test_set = load_match_list(TEST_LIST_PATH)
    target_matches = train_set.union(test_set)

    print(f"Loaded {len(train_set)} Train matches and {len(test_set)} Test matches.")

    # Search for match directories
    match_dirs = [d for d in DATA_DIR.rglob("header*") if d.is_file()]
    unique_match_dirs = list(set([h.parent for h in match_dirs]))

    processed_count = 0
    
    for match_dir in unique_match_dirs:
        match_id = match_dir.name
        
        # Check if match is in our target splits
        if match_id not in target_matches:
            continue
            
        # Determine output path based on split assignment
        if match_id in train_set:
            out_filepath = TRAIN_OUT_DIR / f"{match_id}.csv"
        else:
            out_filepath = TEST_OUT_DIR / f"{match_id}.csv"

        # Skip if already processed (optional, good for resuming)
        if out_filepath.exists():
            print(f"Skipping {match_id} - already exists.")
            continue

        print(f"Processing Match: {match_id}...", end=" ")
        
        success = process_match(match_dir, out_filepath)
        if success:
            processed_count += 1
            print("Done.")
        else:
            print("Skipped (Missing channels/Empty data).")

    print(f"\nPipeline Complete. Successfully processed {processed_count} matches.")

if __name__ == "__main__":
    main()

Loaded 96 Train matches and 23 Test matches.
Processing Match: 1e5341d1-b28f-49b0-84bc-0649d3ed60f8... Done.
Processing Match: ed21c164-6a69-429c-bbf5-27eb555decd7... Done.
Processing Match: c9c3025a-153a-454d-b46e-db10dba9310c... Done.
Processing Match: 62c12fe8-0ee7-43de-b0d6-3d4e16aabdba... Done.
Processing Match: 316c5043-7c0e-4de2-9669-d31bb4aae15a... Done.
Processing Match: 72cd9753-f003-4f0e-b229-fbbe992d8c19... Done.
Processing Match: 7e1ca06d-142d-47dd-af36-f98c773d851f... Done.
Processing Match: 5de6195c-5b14-4234-8670-ab5e72486a02... Done.
Processing Match: 8522d89d-3f5a-45de-94c2-598929025561... Done.
Processing Match: 3aedbbf9-1227-4c69-b493-3f3a43e7ddec... Done.
Processing Match: 94815e7f-b8ed-4859-9ad9-644f6765cda2... Done.
Processing Match: 70990aa3-5510-4a85-90a0-23da894f4a74... Done.
Processing Match: 476a40c0-7822-42ec-9fdf-f7e4ec6895f0... Done.
Processing Match: 098c1484-25ce-484a-a3bb-a884646abffd... Done.
Processing Match: 6fa49016-4ef3-4ebd-beca-673845b0923f... D